In [1]:
import csv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
from torchvision import datasets
import os

In [2]:
import torch.nn as nn
from torchvision import models

class VGG16(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        num_features = self.model.classifier[6].in_features
        self.model.classifier[6] = nn.Linear(num_features, 2) 
        
    def forward(self, x):
        return self.model(x)

In [3]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder("/kaggle/input/fakeface-train-data-v2", transform=transform)
val_dataset   = datasets.ImageFolder("/kaggle/input/fakeface-valid-data-v2", transform=transform)
test_dataset = datasets.ImageFolder("/kaggle/input/fakeface-test-data-v2", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)
test_loader   = DataLoader(test_dataset, batch_size=64, shuffle=False,num_workers=2)

In [4]:
import time
import torch
import torch.nn as nn
from torchvision import models
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

teacher = VGG16() 
teacher = nn.DataParallel(teacher)
teacher = teacher.to(device)

optimizer = torch.optim.Adam(teacher.parameters(), lr=5e-5)
criterion = nn.CrossEntropyLoss()

epoch_times = []
start_training = time.time()

for epoch in range(10):
    start_epoch = time.time()
    
    teacher.train()
    total_loss = 0
    total_batches = len(train_loader)
    
    for batch_idx, (x, y) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False), start=1):
        x, y = x.to(device), y.to(device)
        loss = criterion(teacher(x), y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()
    
    avg = total_loss / len(train_loader)

    teacher.eval()
    correct, total = 0, 0
    all_labels, all_probs = [], [] 
    
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            outputs = teacher(x)
            probs = torch.softmax(outputs, dim=1)[:, 1] 
            
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

            all_labels.extend(y.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            
    val_acc = correct / total
    val_auc = roc_auc_score(all_labels, all_probs)

    epoch_time = time.time() - start_epoch
    epoch_times.append(epoch_time)
    
    print(f"[Teacher] Epoch {epoch+1} | Train Loss: {avg:.4f} | Val Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f} | Time: {epoch_time:.2f}s")

total_time = time.time() - start_training
avg_time = sum(epoch_times) / len(epoch_times)

print(f"\nTotal training time: {total_time:.2f}s")                     
print(f"Average time per epoch: {avg_time:.2f}s")

torch.save(teacher.state_dict(), "/kaggle/working/vgg16.pth")

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100%|██████████| 528M/528M [00:03<00:00, 177MB/s]


[Teacher] Epoch 1 | Train Loss: 0.1769 | Val Acc: 0.9597 | Val AUC: 0.9937 | Time: 1341.60s


[Teacher] Epoch 2 | Train Loss: 0.0256 | Val Acc: 0.9561 | Val AUC: 0.9943 | Time: 1348.63s


[Teacher] Epoch 3 | Train Loss: 0.0147 | Val Acc: 0.9672 | Val AUC: 0.9951 | Time: 1347.30s


[Teacher] Epoch 4 | Train Loss: 0.0130 | Val Acc: 0.9686 | Val AUC: 0.9953 | Time: 1346.24s


...

[Teacher] Epoch 6 | Train Loss: 0.0103 | Val Acc: 0.9720 | Val AUC: 0.9971 | Time: 1344.08s


[Teacher] Epoch 7 | Train Loss: 0.0075 | Val Acc: 0.9708 | Val AUC: 0.9963 | Time: 1344.83s


[Teacher] Epoch 8 | Train Loss: 0.0068 | Val Acc: 0.9741 | Val AUC: 0.9969 | Time: 1343.44s


[Teacher] Epoch 9 | Train Loss: 0.0082 | Val Acc: 0.9742 | Val AUC: 0.9973 | Time: 1343.36s


[Teacher] Epoch 10 | Train Loss: 0.0071 | Val Acc: 0.9709 | Val AUC: 0.9967 | Time: 1342.19s

Total training time: 13445.44s
Average time per epoch: 1344.54s


In [5]:
from sklearn.metrics import classification_report

def evaluate_teacher(model, dataloader, device='cuda'):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            y = y.cpu().numpy() 
            outputs = model(x)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(y)

    print("\n=== Classification Report ===")
    print(classification_report(all_labels, all_preds, digits=4))

In [6]:
evaluate_teacher(teacher, test_loader)


=== Classification Report ===
              precision    recall  f1-score   support

           0     0.9619    0.9831    0.9724     20000
           1     0.9827    0.9610    0.9718     20000

    accuracy                         0.9721     40000
   macro avg     0.9723    0.9721    0.9721     40000
weighted avg     0.9723    0.9721    0.9721     40000

